## BLC data Ingestion Framework
NoteBook: Silver_ingestion_rearc

| Author | Date | Change Record | Description |
| ----------- | ----------- | ----------- | -----------
|Hemant kaushik	|05-Nov-2026|	-	| Notebook for ingesting BLC data from BRNZ to SLVR
 



In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col,trim,current_timestamp,input_file_name,when,row_number,explode,trim,from_json
from pyspark.sql.window import Window
from pyspark.sql.types import StructType,StructField,StringType,DoubleType,LongType,ArrayType

In [0]:
base_path = "/Volumes/workspace/hemant_poc/volume_dev/sourcefiles/"
brnz_prefix = "workspace.bronze.bronze_sql_pr_"
slvr_prefix = "workspace.silver.silver_sql_pr_"

In [0]:
silver_config = [
    {
        "bronze_table": f"{brnz_prefix}series",
        "silver_table": f"{slvr_prefix}series",
        "key_columns": ["series_id"]
    },
    {
        "bronze_table": f"{brnz_prefix}sector",
        "silver_table": f"{slvr_prefix}sector",
        "key_columns": ["sector_code"]
    },
    {
        "bronze_table": f"{brnz_prefix}seasonal",
        "silver_table": f"{slvr_prefix}seasonal",
        "key_columns": ["seasonal_code"]
    },
    {
        "bronze_table": f"{brnz_prefix}period",
        "silver_table": f"{slvr_prefix}period",
        "key_columns": ["period"]
    },         
    {
        "bronze_table": f"{brnz_prefix}measure",
        "silver_table": f"{slvr_prefix}measure",
        "key_columns": ["measure_code"]
    },
    {
        "bronze_table": f"{brnz_prefix}footnote",
        "silver_table": f"{slvr_prefix}footnote",
        "key_columns": ["footnote_code"]
    },    
    {
        "bronze_table": f"{brnz_prefix}duration",
        "silver_table": f"{slvr_prefix}duration",
        "key_columns": ["duration_code"]
    },    
    {
        "bronze_table": f"{brnz_prefix}class",
        "silver_table": f"{slvr_prefix}class",
        "key_columns": ["class_code"]
    },    
    {
        "bronze_table": f"{brnz_prefix}data",
        "silver_table": f"{slvr_prefix}data",
        "key_columns": ["series_id","year","period"]
    }
]

In [0]:
def create_silver_table(
    bronze_table,
    silver_table,
    keys
):

    source_df = spark.table(bronze_table)
    # -----------------------------------
    # 1. Null-key DQ condition
    # -----------------------------------

    key_not_null_condition = " AND ".join([f"`{key}` IS NOT NULL" for key in keys])
    print("-----------------------------------------------")
    print("key_not_null_condition: ",key_not_null_condition)

    # ==================================
    # Business-key partition
    # ==================================

    partition_columns = ", ".join([f"`{key}`" for key in keys ])

    # -----------------------------------
    # 2. Temporary incoming view
    # -----------------------------------

    source_view = (silver_table.replace(".", "_")+ "_incoming")
    print("source_view: ",source_view)

    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW {source_view}
        AS
        SELECT * EXCEPT (_rn)
            FROM
            (
                SELECT
                    *,
                    ROW_NUMBER() OVER (
                        PARTITION BY {partition_columns}
                        ORDER BY
                            _source_modified_at DESC,
                            _ingested_at DESC
                    ) AS _rn
                    FROM {bronze_table}
                    WHERE {key_not_null_condition}
            )
            WHERE _rn = 1
    """)


    # -----------------------------------
    # 3. Silver doesn't exist
    # -----------------------------------

    if not spark.catalog.tableExists(silver_table):

        spark.sql(f"""
            CREATE TABLE {silver_table}
            USING DELTA
            AS
            SELECT *
            FROM {source_view}
        """)

        print(f"Created silver table name : {silver_table}")

        return


    # -----------------------------------
    # 4. Dynamic MERGE condition
    # -----------------------------------

    merge_condition = " AND ".join([
        f"t.`{key}` <=> s.`{key}`"
        for key in keys
    ])
    print(f"merge_condition for {silver_table}:- ",merge_condition)

    # -----------------------------------
    # 5. Columns that actually matter
    # -----------------------------------

    ignore_columns = (
        keys
        + [
            "_source_file",
            "_source_modified_at",
            "_ingested_at"
        ]
    )

    compare_columns = [
        column
        for column in source_df.columns
        if column not in ignore_columns
    ]


    # -----------------------------------
    # 6. Detect true business changes
    # -----------------------------------

    change_condition = " OR ".join([
        f"NOT (t.`{column}` <=> s.`{column}`)"
        for column in compare_columns
    ])

    print(f"change_condition for {silver_table} : ",change_condition)
    # -----------------------------------
    # 7. MERGE
    # -----------------------------------

    spark.sql(f"""
        MERGE INTO {silver_table} AS t
        USING {source_view} AS s
        ON {merge_condition}
        WHEN MATCHED
        AND (
            {change_condition}
        )
        THEN UPDATE SET *
        WHEN NOT MATCHED
        THEN INSERT *
    """)

    print(f"Merged: {silver_table}")
    print("--------------------------------------------------------")

In [0]:
for config in silver_config:

    create_silver_table(
        bronze_table=config["bronze_table"],
        silver_table=config["silver_table"],
        keys=config["key_columns"]
    )

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.silver.silver_sql_pr_population
(
    nation STRING,
    nation_id STRING,
    population DOUBLE,
    year INT,
    source_file STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA;

MERGE INTO workspace.silver.silver_sql_pr_population AS target
USING (
    WITH parsed AS (
        SELECT
            from_json(
                raw_json,
                'STRUCT<data: ARRAY<STRUCT<
                    `Nation`: STRING,
                    `Nation ID`: STRING,
                    `Population`: DOUBLE,
                    `Year`: STRING
                >>>'
            ) AS json_data,
            _source_file
        FROM workspace.bronze.bronze_sql_pr_population
    ),
    exploded AS (
        SELECT
            EXPLODE(json_data.data) AS record,
            _source_file
        FROM parsed
    ),

    cleaned AS (
        SELECT
            TRIM(record.`Nation`) AS nation,
            TRIM(record.`Nation ID`) AS nation_id,
            record.`Population` AS population,
            CAST(record.`Year` AS INT) AS year,
            _source_file AS source_file
        FROM exploded
    )

    SELECT DISTINCT *
    FROM cleaned

) AS source

ON target.nation_id = source.nation_id
AND target.year = source.year


WHEN MATCHED AND (
        NOT (target.nation <=> source.nation)
    OR NOT (target.population <=> source.population)
)
THEN UPDATE SET
    target.nation      = source.nation,
    target.population  = source.population,
    target.source_file = source.source_file,
    target.updated_at  = current_timestamp()


WHEN NOT MATCHED THEN
INSERT (
    nation,
    nation_id,
    population,
    year,
    source_file,
    created_at,
    updated_at
)
VALUES (
    source.nation,
    source.nation_id,
    source.population,
    source.year,
    source.source_file,
    current_timestamp(),
    current_timestamp()
);